# Exercise 1 — Prompt Chaining for a Customer Support AI

**Tools used:** ChatGPT for prompt/code design, Google Colab for Python execution, GitHub for notebook sharing.

**Goal:** Build a simple multi-step customer-support flow where each step uses the output from the prior step.

### Prompt iteration
**First prompt (too vague):**  
> Create a Python customer-support program.

**Improved prompt:**  
> Create a Python customer-support prompt chain with four stages: (1) classify the customer's issue, (2) identify missing information needed to help, (3) propose a solution using the classification and collected information, and (4) decide whether the case should be escalated. Each stage must receive the previous stage's output. Use a professional, concise tone. Do not request passwords, full card numbers, or other sensitive information. Return simple Python dictionaries so the flow is easy to inspect in Colab. Include one sample customer case and print every stage of the chain.

This revision is more specific because it defines the number of steps, the dependency between steps, the output format, tone, privacy constraints, and required test output.

## Prompts used by step

**Step 1 — Classify issue**  
> You are a customer-support classifier. Classify the message as Billing, Shipping, Technical, or General. Also assign urgency as Low, Medium, or High. Return only the category and urgency.

**Step 2 — Gather missing information**  
> Using the classification from Step 1 and the original customer message, identify only the minimum non-sensitive information still needed to solve the issue. Do not ask for passwords or full payment-card numbers.

**Step 3 — Propose solution**  
> Using the Step 1 classification and the information collected in Step 2, propose a concise next action for the customer. Do not promise a refund or resolution unless the available information supports it.

**Step 4 — Escalation rule**  
> Review the issue classification and proposed solution. Escalate only if there is a fraud/security concern, a high-value billing issue, a high-urgency case, or the proposed solution cannot resolve the issue. Return the escalation decision and reason.

In [1]:
# Sample customer message used to test the chain
customer_message = "I think I was charged twice for the same sandwich order and I want it fixed."


def classify_issue(message):
    text = message.lower()
    if any(word in text for word in ["charged", "refund", "payment", "billing"]):
        category = "Billing"
    elif any(word in text for word in ["delivery", "shipping", "late", "arrive"]):
        category = "Shipping"
    elif any(word in text for word in ["error", "login", "app", "website"]):
        category = "Technical"
    else:
        category = "General"

    urgency = "High" if any(word in text for word in ["fraud", "stolen", "unauthorized"]) else "Medium"
    return {"category": category, "urgency": urgency}


def gather_missing_info(classification, message):
    # This step depends on Step 1's classification.
    if classification["category"] == "Billing":
        return {
            "needed": ["order number", "duplicate charge status (pending or posted)", "duplicate amount"],
            "avoid": ["password", "full card number"]
        }
    if classification["category"] == "Shipping":
        return {"needed": ["order number", "expected delivery date"], "avoid": ["password"]}
    if classification["category"] == "Technical":
        return {"needed": ["device/browser", "exact error message"], "avoid": ["password"]}
    return {"needed": ["short description of the requested help"], "avoid": ["password"]}


def propose_solution(classification, collected_info):
    # This step uses the category from Step 1 and the information gathered after Step 2.
    if classification["category"] == "Billing":
        if collected_info.get("duplicate_charge_status") == "both posted":
            return {
                "action": "Verify the two posted transactions against the same order and submit a duplicate-charge correction if the duplicate is confirmed.",
                "status": "actionable"
            }
        return {
            "action": "First verify whether the second charge is pending or posted before requesting a billing correction.",
            "status": "needs verification"
        }
    return {"action": "Use the collected information to continue standard support troubleshooting.", "status": "actionable"}


def decide_escalation(classification, collected_info, solution):
    # This final step depends on the earlier classification and solution.
    amount = float(collected_info.get("duplicate_amount", 0) or 0)
    fraud_signal = collected_info.get("fraud_or_unauthorized", False)
    high_value = amount >= 100
    unresolved = solution["status"] != "actionable"
    high_urgency = classification["urgency"] == "High"

    escalate = fraud_signal or high_value or unresolved or high_urgency
    if fraud_signal:
        reason = "Possible fraud or unauthorized charge."
    elif high_value:
        reason = "Billing amount meets the escalation threshold used in this simulation."
    elif high_urgency:
        reason = "Issue was classified as high urgency."
    elif unresolved:
        reason = "The case still needs verification before it can be resolved."
    else:
        reason = "The case can continue through the standard support flow."
    return {"escalate": escalate, "reason": reason}


# Step 1
classification = classify_issue(customer_message)

# Step 2
missing_info = gather_missing_info(classification, customer_message)

# Simulated customer follow-up after Step 2 asks for the missing information
collected_info = {
    "order_number": "A1024",
    "duplicate_charge_status": "both posted",
    "duplicate_amount": 24.50,
    "fraud_or_unauthorized": False
}

# Step 3
solution = propose_solution(classification, collected_info)

# Step 4
escalation = decide_escalation(classification, collected_info, solution)

print("Customer message:", customer_message)
print("\nStep 1 - Classification:", classification)
print("Step 2 - Missing information requested:", missing_info)
print("Step 2 - Collected information:", collected_info)
print("Step 3 - Proposed solution:", solution)
print("Step 4 - Escalation decision:", escalation)


Customer message: I think I was charged twice for the same sandwich order and I want it fixed.

Step 1 - Classification: {'category': 'Billing', 'urgency': 'Medium'}
Step 2 - Missing information requested: {'needed': ['order number', 'duplicate charge status (pending or posted)', 'duplicate amount'], 'avoid': ['password', 'full card number']}
Step 2 - Collected information: {'order_number': 'A1024', 'duplicate_charge_status': 'both posted', 'duplicate_amount': 24.5, 'fraud_or_unauthorized': False}
Step 3 - Proposed solution: {'action': 'Verify the two posted transactions against the same order and submit a duplicate-charge correction if the duplicate is confirmed.', 'status': 'actionable'}
Step 4 - Escalation decision: {'escalate': False, 'reason': 'The case can continue through the standard support flow.'}


## What the test demonstrates
The classification from Step 1 determines what Step 2 asks for. The collected information is then used by Step 3 to choose an appropriate action. Step 4 reviews the earlier outputs to decide whether escalation is necessary. This makes the stages a true chain instead of four unrelated prompts.